Nelle reti cellulari bisogna distinguere tre soggetti:
- **Terminale e SIM**: sarebbe il dispositivo dell'utente e il componente che contiene la sua identità e segreto di autenticazione.
- **Serving Network**: la rete alla quale il dispositivo è attualmente collegato
- **Home Network**: la rete dell'operatore con cui l'utente ha il contratto e che ha fornito la SIM

Ad esempio con il roaming, se sono cliente di un ISP italiano ma vado all'estero mi collego allora alla rete di un operatore locale. La rete visitata deve poter verificare che sono abbonato autorizzato, anche se **la chiave segreta della SIM è condivisa solo con l'operatore di origine**.

Quindi per questa sezione ci chiediamo: **come può la rete visitata autenticare l'utente senza ricevere la sua chiave segreta?**  
La soluzione sarà farle ricevere dalla home network materiale utilizzabile per specifiche autorizzazioni.

Distinguiamo alcuni servizi, per capire quanto segue:
- il **VLR** (Visitor Location Register) è un database della rete visitata che contiene informazioni sugli utenti attualmente connessi. Appartiene quindi alla serving network.
- L'**AuC** (Authentication Center) è un database della home network che gestisce il materiale segreto necessario all'autenticazione
- l'**HLR** (Home Location Register) è un database della home network che contiene informazioni sugli abbonati

Le denominazioni cambiano tra le diverse generazioni di reti cellulari, ma il concetto rimane lo stesso. 

Prima della parte tecnica accenniamo all'evoluzione della sicurezza nell'arco delle generazioni.

Nel **1G**, si utilizzava un sistema analogico con protezione molto limitata: le comunicazioni radio potevano essere intercettate senza le difese crittografiche introdotte nelle generazioni successive.

Con il **2G/GSM** si introdusse l'autenticazione dell'abbonato e la cifratura delle comunicazioni radio. Tuttavia ancora tre limitazioni grosse:
1. alcune costruzioni crittografiche diffuse erano mantenute segrete, quindi non potevano essere verificate da terzi. Questo è un problema perché se la costruzione è debole, non c'è modo di saperlo.
2. l'autenticazione era unilaterale: l'utente si autentica verso la rete, ma la rete non si autentica verso l'utente. Questo permette attacchi di tipo man-in-the-middle.
3. la sicurezzza della rete interna non riceveva una protezione paragonabile a quella delle generazioni successive

Nel **3G** vengono introdotti miglioramenti fondamentali: autenticazione mutua, algoritmi crittografici pubblici, protezione dei collegamenti interni e uso di chiavi distinte per cifratura e integrità. 

Nel **4G** la sicurezza è affrontata in modo più sistematico, considerando le diverse parti dell'architettura e i collegamenti tra esse. 

**Il punto è che la sicurezza di una rete non dipende solo dalla robustezza del cifrario radio: occorre proteggere autenticazione, infrastruttura e relazioni tra le diverse entità.** Questo è chiaro guardando all'architettura **5G**, esistono più domini da proteggere!

<img src="imgs/domains.png" width="500">

Il punto più importante qui è **Visibility and Configuration of Security**, in pratica **un sistema può supportare una protezione senza usarla effettivamente in una certa connessione --> bisogna distinguere capacità dichiarate, configurazione e protezioni attivate!**  
(es. un terminale può supportare la cifratura, ma la rete visitata può decidere di non attivarla per una certa connessione)

Vediamo ora alcune delle componenti principali della sicurezza nelle reti cellulari:

<img src="imgs/5G_security.png" width="500">

Il punto concettuale qui è che non tutti i componenti devono possedere gli stessi segreti: la chiave segreta della SIM ad esempio deve essere custodita in un dominio più protetto rispetto a un apparato radio distribuito sul territorio.

(N.B. questa è una parte introduttiva, non è necessario studiarla per l'esame ma utile per capire il contesto. La parte da studiare direttamente comincia ora, con l'autenticazione)

### Autenticazione
Si parla ora dell'autenticazione dell'abbonato alla serving network.

Quando il dispositivo si registra nella rete, o in altre circostanze previste dal sistema, la rete può avviare l'autenticazione dell'abbonato.  
Il percorso semplificato è:
1. Il dispositivo contatta la rete e comunica informazioni utili ad identificarlo;
2. la rete esegue l'autenticazione;
3. vengono rese disponibili le chiavi di sessione necessarie per cifratura e integrità dei messaggi;
4. viene attivata la protezione delle comunicazioni prevista dal sistema

Importante sempre ribadire che autenticare e cifrare **sono operazioni diverse**: l'autenticazione verifica il possesso di un segreto mentre la cifratura protegge le comunicazioni successive. In particolare come vedremo il protocollo produce chiavi temporanee per quelle comunicazioni. 

#### Autenticazione unilaterale del 2G
Ogni abbonato ha una **chiave segreta permanente $K_i$**, memorizzata nella SIM e nell'AuC della home network. Non rappresenta la password dell'utente o il PIN della SIM, ma un segreto crittografico associato all'abbonamento.  
L'**IMSI** invece **identifica l'abbonato** (non è l'IMEI né il numero telefonico).  

L'autenticazione usa il modello challenge-response: la rete invia una challenge casuale $RAND$ al terminale e la SIM calcola $SRES = A_3(K_i, RAND)$, dove $SRES$ significa **Signed Response** (nel contesto GSM è il nome della risposta, non una firma digitale a chiave pubblica).  
A quel punto la rete verifica se la risposta ricevuta coincide con quella attesa.

Parlando di visited network, la rete visitata non conosce $K_i$ e quindi non può calcolare $SRES$, come fare?  
Semplice: la rete visitata manda la challenge alla SIM e riceve la risposta, ma per verificarla chiede alla home network di calcolare $SRES$ e farsi restituire il risultato.

La procedura è **unilaterale** perché la SIM non riceve una corrispondente prova di autenticità della rete.

In pratica ciò è implementato tramite dei **triplet di autenticazione** (detto **authentication vector**): si tratta di valori preparati dalla home network della forma:
$$(RAND, XRES, K_c)$$
dove $RAND$ è la challenge da inviare alla SIM, $XRES$ è la risposta attesa (Si scrive $XRES$ per distinguerla dalla $SRES$ risposta effettiva della SIM (la tripla ricordiamo infatti è prodotta dalla home network, non dalla SIM))e $K_c$ è la chiave di cifratura da usare per le comunicazioni successive.

La rete visitata comunica l'identità dell'abbonato alla home network e riceve un insieme di $N$ triplet. In questo modo per una successiva autenticazione la rete visitata può usare uno dei triplet ricevuti senza dover contattare la home network ogni volta.

I vantaggi nell'uso dei triplet sono due:
1. **Efficienza**: non serve contattare la home network per ogni autenticazione
2. **Casualità**; è la home network ad occuparsi di generare le challenge, anziché affidarsi alla qualità del generatore della rete visitata

Per generare $SRES$ è $K_c$ si utilizzano due funzioni distinte ma con stessi input:
$$SRES = A_3(K_i, RAND) \quad K_c = A_8(K_i, RAND)$$
Poiché gli input coincidono le funzioni possono essere realizzate con una costruzione congiunta, detta **A3/A8** o **A38**.

Si ha $K_i$ a 128 bit, $RAND$ a 128 bit, $SRES$ a 32 bit e $K_c$ a 64 bit nominali.  
Chiaramente chiave permanente e chiave di cifratura per le comunicazioni radio sono diverse, ed in particolare per ogni nuova autenticazione si genera un nuovo $K_c$.

Si parla di 64 bit nominali per $K_c$ perché in realtà la chiave viene ridotta a 54 bit effettivi per via di alcune debolezze nelle implementazioni storiche basate su COMP128-1: 10 bit di $K_c$ erano posti a zero.

#### COMP128 e il security by obscurity
Gli algoritmi A3/A8 possono essere specifici dell'operatore in quanto devono essere eseguiti soltanto nei punti che condividono il segreto: SIM e AuC. La rete visitata deve ricevere solo i risultati degli algoritmi.

Si introduce così **COMP128 e il problema del security by obscurity**: cercare di ottenere sicurezza mantenendo segreto il funzionamento dell'algoritmo.

Gli attacchi storici contro COMP128 (rotta nel 1998) hanno mostrato come, su implementazioni vulnerabili, interrogare la SIM con challenge scelte ad hoc potesse permettere di ricostruire la chiave segreta $K_i$. Una volta conosciuto il segreto poi, l'attaccante può calcolare $SRES$ e $K_c$ per qualsiasi challenge, quindi impersonare l'abbonato: si parla di **clonazione della SIM**. Tutta la sicurezza si reggeva sul fatto che gli algoritmi fossero segreti, ma ovviamente (che fosse per reverse engineering o per fuga di informazioni) sono stati scoperti.

Si fa quindi riferimento al **principio di Kerckhoffs**: la sicurezza deve reggere anche quando l'attaccante conosce il funzionamento dell'algoritmo, ciò che deve rimanere segreto è la chiave.

La possibilità di un attaccante di interrogare la SIM e ottenere risposte su challenge scelte stava nella **mancanza di autenticazione bidirezionale nel caso 2G**: chiunque poteva presentarsi come falsa stazione radio (Rogue BTS) e indurre un terminale a interagire con lui. 

### Authentication con 3G: AKA
Dal 3G viene introdotto un nuovo protocollo di autenticazione, detto **Authentication and Key Agreement (AKA)**.  
AKA si pone due obiettivi:
1. autenticare l'abbonato alla rete visitata, come nel caso 2G
2. stabilire il materiale di chiave per proteggere le comunicazioni radio successive

Per la spiegazione si fa riferimento al modello 3G, ma da tenere a mente che il 4G e il 5G ne riprendono le idee introducendo però ulteriori derivazioni di chiavi e differenze nel protocollo. 

Con AKA le novità centrali sono **autenticazione mutua**, e **produzione di chiavi distinte per finalità diverse (integrità e cifratura)**.  
Inoltre altra novità importantissima: **gli algoritmi sono prima condivisi con la comunità di ricerca, e solo dopo scelti per l'utilizzo (l'opposto di security by obscurity)**.

Stavolta la home network produce un **authentication vector** con cinque elementi $AV = (RAND, XRES, CK, IK, AUTN)$, dove:
- $RAND$ è la challenge da inviare alla SIM
- $XRES$ è la risposta attesa della rete dalla SIM. 
- $CK$ Cipher key, è la chiave di cifratura per le comunicazioni radio
- $IK$ Integrity key, è la chiave di integrità per le comunicazioni radio
- $AUTN$ Authentication token, permette alla SIM di verificare l'autenticazione della rete

Nel modello semplificato lo scambio radio è il seguente:  

<img src="imgs/aka.png" width="300">

La SIM prima verifica $AUTN$, poi produce la risposta $RES = f_2(K_i, RAND)$ (lunga da 32 a 128 bit) e la invia alla rete. La rete confronta $RES$ con $XRES$ e se coincidono l'autenticazione è completata.

Le altre funzioni producono:
$$CK = f_3(K_i, RAND) \quad IK = f_4(K_i, RAND)$$
entrambe lunghe 128 bit.

Quindi le chiavi di cifratura e integrità vengono ottenute da entrambe le parti senza essere trasmesse sul collegamento radio. La SIM le calcola, mentre la rete visitatwa le riceve dalla home network attraverso l'infrastruttura di rete.

Usare chiavi distinte per cifratura e integrità è importante perché si separi la protezione dalla confidenzialità (nessuno legge i messaggi scambiati) da quella di integrità (nessuno può modificare i messaggi scambiati senza essere scoperto).



#### Come autentica la rete il terminale? 
Per autenticare la rete il terminale potrebbe inviarle una propria challenge casuale e pretendere una risposta autenticata. Tuttavia questa soluzione ha un **costo significativo** dato che richiederebbe un altro passaggio nello scambio radio.

AKA usa invece un **numero di sequenza** $SQN$, e uno stato conservato dalla SIM. 

In particolare la home network mantiene un contatore (numero di sequenza) per ogni abbonato, che viene incrementato ad ogni autenticazione. La SIM invece conserva il contatore su tutte le autenticazioni che ha effettuato in precedenza (anche tra diversi operatori visitati).

In questo modo la SIM può verificare che la rete stia usando un numero di sequenza valido.  
In particolare la SIM conserva uno stato indicato come $SQN_{MS}$, e riceve dalla rete un token contenente un altro numero $SQN$. Nel modello più semplice la SIM controlla 
$$SQN = SQN_{MS} + 1$$
Nella realtà però ricordiamo che la home network genera più challenge in parallelo per la questione vista prima e quindi diversi numeri di sequenza possono restare inutilizzati, per non parlare di eventuali messaggi persi.  
Per questi modtivi la verifica della SIM è più complessa, e accetta un **range di valori validi** per il numero di sequenza. Se gli stati sono disallineati, è prevista una procedura di risincronizzazione autenticata tra SIM e home network.

**Il numero di sequenza però quindi deve essere anche autenticato**, perché altrimenti un attaccante potrebbe limitarsi a inventare un numero nel range di quelli attesi dalla SIM.

Quindi in generale servono due controlli insieme:
- **il token deve essere autentico**
- **il token deve essere fresco** (es. se la SIM ha accettato SQN con 101 e un attaccante ripropone quel token, deve rifiutarlo)

(ricorda infatti che la serving network manda al terminale la coppia (RAND, AUTN))

Per fare ciò il token $AUTN$ viene scritto nella forma:
$$AUTN = SQN || AMF || MAC-A$$
dove $SQN$ è un campo a 48 bit per la verifica di freshness, $AMF$ 16 bit di informazioni di gestione dell'autenticazione e $MAC-A$ è un codice di autenticazione del token.

Il MAC-A è calcolato dalla home network come:
$$MAC-A = f_1(K_i, SQN, RAND, AMF)$$
La SIM allora conosce $K_i$ e gli altri valori (li riceve tutti dalla serving network come parte del token $AUTN$, e $RAND$ a parte), dunque può calcolare il $MAC$ e confrontarlo con quello ricevuto. 

In questo modo:
- l'attaccante può scegliere un contatore, ma non può calcolare il MAC-A dato che dipende dal segreto
- non può modificare un token, in quanto il MAC-A è calcolato in funzione di tutti i campi del token AUTN stesso e quindi non coinciderebbe più con il MAC-A ricevuto
- non può riproporre un token già usato, perché la SIM rifiuterebbe il numero di sequenza non più valido

Il $MAC-A$ inoltre lega il token alla challenge $RAND$ di quella autenticazione.

Quindi in questo modello la SIM verifica materiale autentico prodotto dalla home network e fornito alla serving network, senza però che sia necessario nascondere il funzionamento dell'algoritmo, ma solo il segreto $K_i$!

##### Perché nascondere SQN
Il numero di sequenza $SQN$ potrebbe rivelare informazioni utili a distinguere/seguire un utente. Per ridurre questa esposizione, viene mascherato con una **Anonymity Key**:
$$AK = f_5(K_i, RAND)$$
$AK$ è lunga 48 bit, come il numero di sequenza $SQN$. Il formato completo del token è in definitiva:
$$AUTN = (SQN \oplus AK) || AMF || MAC-A$$
Al momento di ricezione, la SIM calcola localmente la stessa $AK$ e recupera il contatore sfruttando la proprietà dell'operazione XOR vista negli stream cipher:
$$SQN = (SQN \oplus AK) \oplus AK$$
calcolo che un attaccante non può fare perché non conosce $K_i$ e quindi non può calcolare $AK$.

Il $MAC-A$ invece continua ad essere calcolato usando il vero $SQN$: la SIM prima lo recupera da $AUTN$ e poi lo usa per calcolare il MAC-A e confrontarlo con quello ricevuto.

##### Riassunto della procedura AKA completa
Nel modello (semplificato) 3G:

<img src="imgs/aka_full.png" width="500">